In [1]:
from pathlib import Path
import json
import warnings
import pickle
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from transformers.utils.import_utils import candidates

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
print("Libraries imported successfully.")
print("LightGBM version:", lgb.__version__)

Libraries imported successfully.
LightGBM version: 4.5.0


## Project Paths

In [3]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
CANDIDATE_DIR = ARTIFACT_DIR / "candidates"
LTR_DIR = ARTIFACT_DIR / "ltr"
LTR_DIR.mkdir(parents=True,exist_ok=True)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("CANDIDATE_DIR:", CANDIDATE_DIR)
print("LTR_DIR      :", LTR_DIR)

PROJECT_ROOT : D:\iPrint-News-Recommendation-Ranking-System\Notebook
DATA_DIR     : D:\iPrint-News-Recommendation-Ranking-System\Notebook\data
CANDIDATE_DIR: D:\iPrint-News-Recommendation-Ranking-System\Notebook\artifacts\candidates
LTR_DIR      : D:\iPrint-News-Recommendation-Ranking-System\Notebook\artifacts\ltr


## Configurations

In [5]:
TARGET_K = 10
RANDOM_STATE = 42
MIN_INTERACTIONS_FOR_PERSONALIZATION = 5
VALIDATION_FRACTION = 0.20
LGB_PARAMS = {"objective": "lambdarank","metric": "ndcg","ndcg_at": [5, 10],"label_gain": [0, 1],"learning_rate": 0.05,"num_leaves": 31,"max_depth": -1,"min_child_samples": 30,"subsample": 0.8,"subsample_freq": 1,"colsample_bytree": 0.8,"reg_alpha": 0.1,"reg_lambda": 1.0,"verbosity": -1,"random_state": RANDOM_STATE,"n_jobs": -1,}
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 50
print(json.dumps(LGB_PARAMS, indent=4))

{
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_at": [
        5,
        10
    ],
    "label_gain": [
        0,
        1
    ],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 30,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbosity": -1,
    "random_state": 42,
    "n_jobs": -1
}


## Load Candidate Artifacts

In [21]:
candidate_features_path = (CANDIDATE_DIR / "candidate_features.parquet")
test_truth_path = (CANDIDATE_DIR / "test_truth.parquet")
raw_candidates_path = (CANDIDATE_DIR / "raw_candidates.parquet")
config_path = (CANDIDATE_DIR / "candidate_generation_config.json")
print("Checking files...")
for path in [candidate_features_path,test_truth_path,raw_candidates_path,]:
    print(f"{path.name}:", "found" if path.exists() else "missing")

Checking files...
candidate_features.parquet: found
test_truth.parquet: found
raw_candidates.parquet: found


In [22]:
from pathlib import Path
CURRENT_DIR = Path.cwd()
print("Current working directory:")
print(CURRENT_DIR)
print("Searching for candidate_features.parquet...")
matches = list(CURRENT_DIR.parent.rglob("candidate_features.parquet"))
matches += list(CURRENT_DIR.rglob("candidate_features.parquet"))
matches = list(dict.fromkeys(matches))
if matches:
    print("Found candidate_features.parquet:")
    for path in matches:
        print("  ", path)
else:
    print("candidate_features.parquet was not found " "under the current directory.")

Current working directory:
D:\iPrint-News-Recommendation-Ranking-System\Notebook
Searching for candidate_features.parquet...
Found candidate_features.parquet:
   D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\candidate_features.parquet


In [24]:
from pathlib import Path
CURRENT_DIR = Path.cwd()
PROJECT_NAME = "iPrint-News-Recommendation-Ranking-System"
def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    search_locations = [start_path,*start_path.parents]
    for path in search_locations:
        if path.name == PROJECT_NAME:
            return path
        if ((path / "Notebook").exists() and path.name.lower().startswith("iprint")):
            return path
    return None
PROJECT_ROOT = find_project_root(CURRENT_DIR)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not automatically locate the "f"'{PROJECT_NAME}' project root.\n" f"Current directory: {CURRENT_DIR}")
print(PROJECT_ROOT)

D:\iPrint-News-Recommendation-Ranking-System


In [25]:
ARTIFACT_DIR = (PROJECT_ROOT / "artifacts")
CANDIDATE_DIR = (ARTIFACT_DIR / "candidates")
LTR_DIR = (ARTIFACT_DIR / "ltr")
LTR_DIR.mkdir(parents=True,exist_ok=True)
print("ARTIFACT_DIR :", ARTIFACT_DIR)
print("CANDIDATE_DIR:", CANDIDATE_DIR)
print("LTR_DIR :",LTR_DIR)

ARTIFACT_DIR : D:\iPrint-News-Recommendation-Ranking-System\artifacts
CANDIDATE_DIR: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates
LTR_DIR : D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr


In [26]:
candidate_files = {"candidate_features": (CANDIDATE_DIR / "candidate_features.parquet"),"test_truth": (CANDIDATE_DIR / "test_truth.parquet"),"raw_candidates": (CANDIDATE_DIR/ "raw_candidates.parquet"),"candidate_config": (CANDIDATE_DIR / "candidate_generation_config.json"),}
print("Candidate Artifact Check")
for name, path in candidate_files.items():
    status = ("found" if path.exists() else "missing")
    print(f"{status:<12} {name:<25} {path}")

Candidate Artifact Check
found        candidate_features        D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\candidate_features.parquet
found        test_truth                D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\test_truth.parquet
found        raw_candidates            D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\raw_candidates.parquet
found        candidate_config          D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\candidate_generation_config.json


In [27]:
for filename in ["candidate_features.parquet", "test_truth.parquet", "raw_candidates.parquet","candidate_features.csv",]:
    print(filename)
    found = list(PROJECT_ROOT.rglob(filename))
    if found:
        for path in found:
            print("path")
    else:
        print("not found")

candidate_features.parquet
path
test_truth.parquet
path
raw_candidates.parquet
path
candidate_features.csv
path


## Inspect Candidates Features

In [36]:
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "iPrint-News-Recommendation-Ranking-System":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find project root: ", "iPrint-News-Recommendation-Ranking-System")
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Project Root:", PROJECT_ROOT)
CANDIDATE_DIR = PROJECT_ROOT / "artifacts" / "candidates"
print("Candidate Directory:", CANDIDATE_DIR)
candidate_features_path = CANDIDATE_DIR / "candidate_features.parquet"
test_truth_path = CANDIDATE_DIR / "test_truth.parquet"
raw_candidates_path = CANDIDATE_DIR / "raw_candidates.parquet"
required_files = {"candidate_features": candidate_features_path, "test_truth": test_truth_path,"raw_candidates": raw_candidates_path,}
print("\nChecking candidate artifacts...")
for name, path in required_files.items():
    print(f"{name:20s}: {path}")
    print(f"{'Exists':20s}: {path.exists()}")
    print()
missing_files = [str(path) for path in required_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError("\nMissing candidate-generation artifacts:\n" + "\n".join(f"  - {path}" for path in missing_files) + "\n\nRun Candidate_Generation.ipynb first and make sure " "the artifacts are saved under:\n" f"{CANDIDATE_DIR}")
candidate_features = pd.read_parquet(candidate_features_path)
test_truth = pd.read_parquet(test_truth_path)
raw_candidates = pd.read_parquet(raw_candidates_path)
print("Candidate artifacts loaded successfully.")
print("nShapes:")
print("candidate_features:", candidate_features.shape)
print("test_truth:", test_truth.shape)
print("raw_candidates:", raw_candidates.shape)

Project Root: D:\iPrint-News-Recommendation-Ranking-System
Candidate Directory: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates

Checking candidate artifacts...
candidate_features  : D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\candidate_features.parquet
Exists              : True

test_truth          : D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\test_truth.parquet
Exists              : True

raw_candidates      : D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\raw_candidates.parquet
Exists              : True

Candidate artifacts loaded successfully.
nShapes:
candidate_features: (514172, 12)
test_truth: (1715, 2)
raw_candidates: (924719, 10)


In [38]:
CANDIDATE_DIR = PROJECT_ROOT / "artifacts" / "candidates"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
raw_candidates.to_parquet(CANDIDATE_DIR / "raw_candidates.parquet", index=False)
candidate_features.to_parquet(CANDIDATE_DIR / "candidate_features.parquet", index=False)
candidate_features.to_csv(CANDIDATE_DIR / "candidate_features.csv", index=False)
test_truth.to_parquet(CANDIDATE_DIR / "test_truth.parquet", index=False)
print("Candidate artifacts saved to:")
print(CANDIDATE_DIR)

Candidate artifacts saved to:
D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates


## Load Candidate Generation Configurations

In [39]:
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        candidate_config = json.load(f)
    print(json.dumps(candidate_config, indent=4, default=str))
else:
    candidate_config = {}
    print("candidate_generation_config.json not found.")

{
    "candidate_k_per_source": 100,
    "final_candidate_k": 300,
    "top_k": 10,
    "recent_days": 7,
    "recency_half_life_days": 14,
    "interaction_weights": {
        "content_watched": 1.0,
        "content_liked": 2.0,
        "content_saved": 3.0,
        "content_followed": 4.0,
        "content_commented_on": 5.0
    },
    "candidate_sources": [
        "popularity",
        "trending",
        "country",
        "tfidf",
        "item_cf",
        "als",
        "semantic"
    ],
    "content_based_language": "en",
    "availability_rule": "latest_content_state",
    "seen_item_rule": "remove_training_history_items",
    "split_strategy": "last_interaction_per_user"
}


## Basic Validations

In [42]:
required_base_columns = ["consumer_id", "item_id",]
missing_base_columns = [col for col in required_base_columns  if col not in candidate_features.columns]
if missing_base_columns:
    raise ValueError("Missing required columns: "  f"{missing_base_columns}")
print("Base columns available")
print("Unique users:", candidate_features["consumer_id"].nunique())
print("Unique candidate articles:", candidate_features["item_id"].nunique())
print("Total candidate rows:", len(candidate_features))

Base columns available
Unique users: 1715
Unique candidate articles: 2901
Total candidate rows: 514172


## Normalize ID

In [44]:
candidate_features["consumer_id"] = (candidate_features["consumer_id"].astype(str).str.strip())
candidate_features["item_id"] = (candidate_features["item_id"].astype(str).str.strip())
raw_candidates["consumer_id"] = (raw_candidates["consumer_id"].astype(str).str.strip())
raw_candidates["item_id"] = (raw_candidates["item_id"].astype(str).str.strip())
if "consumer_id" in test_truth.columns:
    test_truth["consumer_id"] = (test_truth["consumer_id"].astype(str).str.strip())
if "item_id" in test_truth.columns:
    test_truth["item_id"] = (test_truth["item_id"].astype(str).str.strip())
print("IDs normalized")

IDs normalized


In [46]:
display(test_truth.head(20))
print("test_truth columns:", test_truth.columns.tolist())

,consumer_id,item_id
0,-1007001694607905623,8729086959762650511
1,-1032019229384696495,3566197569262766169
2,-108842214936804958,-4029704725707465084
3,-1093393486211919385,-3191013159715472435
4,-1110220372195277179,-1297580205670251233
5,-1113322110177216831,-5953227649059336919
6,-1119397949556155765,-2402288292108892893
7,-1130272294246983140,8477804012624580461
8,-1157793243708252831,-6872546942144599345
9,-1160159014793528221,-3462051751080362224


test_truth columns: ['consumer_id', 'item_id']


## 1. Build Held Out Truth Mappings

In [48]:
if {"consumer_id", "item_id"}.issubset(test_truth.columns):
    test_truth_map = (test_truth.groupby("consumer_id")["item_id"].apply(set).to_dict())
else:
    raise ValueError("test_truth must contain consumer_id and item_id.")
print("Users with held-out truth:", len(test_truth_map))
example_user = next(iter(test_truth_map))
print("Example user:", example_user)
print("Held-out items:", test_truth_map[example_user])

Users with held-out truth: 1715
Example user: -1007001694607905623
Held-out items: {'8729086959762650511'}
